In [1]:
%pip install pandas numpy scikit-learn xgboost lightgbm catboost matplotlib
# This notebook demonstrates a minimal heat prediction workflow.


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Optional: preview the prebuilt feature table if your local pyarrow version can read it.
# The model demo below does not depend on this parquet file; it reads the CSV directly.
import pandas as pd

features_path = "data/features.parquet"
try:
    print(f"Loading optional feature data from {features_path}")
    features_preview = pd.read_parquet(features_path)
    print(f"Loaded {len(features_preview)} rows with columns: {list(features_preview.columns)}")
    display(features_preview.head())
except Exception as exc:
    print(f"Skipping optional parquet preview: {exc}")
    print("Continue to the next cell; the model workflow uses the CSV file directly.")


Loading optional feature data from data/features.parquet
Loaded 71597 rows with columns: ['city', 'temp_c', 'humidity', 'wind_knots', 'date', 'temp_f', 'wind_mph', 'dew_point_c', 'dew_point_f', 'heat_index']


,city,temp_c,humidity,wind_knots,date,temp_f,wind_mph,dew_point_c,dew_point_f,heat_index
0,K3J7,12.50,47.6417,3.681,NaT,54.500,4.236021,1.682022,35.027640,90.190745
1,K4M9,5.50,89.7800,13.508,NaT,41.900,15.544736,3.956554,39.121797,108.343769
2,KABI,13.08,52.9500,9.702,NaT,55.544,11.164868,3.703296,38.665933,89.036259
3,KADH,1.00,74.6130,4.169,NaT,33.800,4.797602,-2.996717,26.605910,156.458327
4,KAKO,8.74,35.1528,9.072,NaT,47.732,10.439876,-5.813739,21.535270,94.662941


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# Optional boosting libraries that may not be installed/loadable in every environment.
try:
    import xgboost as xgb
    xgboost_available = True
except Exception as exc:
    xgboost_available = False
    print("Warning: XGBoost not available in this environment. Skipping XGBoost model.")
    print(str(exc))
    print("On Mac, you may need to install libomp: brew install libomp")

try:
    import lightgbm as lgb
    lightgbm_available = True
except Exception as exc:
    lightgbm_available = False
    print("Warning: LightGBM not available in this environment. Skipping LightGBM model.")
    print(str(exc))

try:
    import catboost as cb
    catboost_available = True
except Exception as exc:
    catboost_available = False
    print("Warning: CatBoost not available in this environment. Skipping CatBoost model.")
    print(str(exc))

# Configurable CSV pattern and city filter (ICAO location identifier).
from pathlib import Path

csv_pattern = "data/daily-weather-rice_*.csv"
city_filter = "KQRH"  # change this to the desired city's ICAO identifier

# Load and combine all matching real daily weather CSV files.
csv_paths = sorted(Path("data").glob("daily-weather-rice_*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found matching {csv_pattern}")

df_raw = pd.concat((pd.read_csv(path) for path in csv_paths), ignore_index=True)
print(f"Loaded {len(df_raw)} rows from {len(csv_paths)} CSV files")
for path in csv_paths:
    print(f"- {path}")

# Normalize raw columns into the units the model expects.
df = pd.DataFrame({
    "city": df_raw["CITY_LOCATION_IDENTIFIER__UP_TO_9_ALPHANUMERIC_CHARACTERS_"],
    "temp_f": df_raw["AVERAGE_TEMPERATURE_C___FLOAT_VALUE_TO_NEAREST_HUNDREDTHS_PLACE"] * 9 / 5 + 32,
    "humidity": df_raw["AVERAGE_RELATIVE_HUMIDITY_____FLOAT_VALUE_TO_NEAREST_HUNDREDTHS_PLACE"],
    "wind_mph": df_raw["AVERAGE_WIND_SPEED_KNOTS___FLOAT_VALUE_TO_NEAREST_HUNDREDTHS_PLACE"] * 1.15078,
})

# Compute heat index from temp/humidity using the NOAA Rothfusz regression.
t, rh = df["temp_f"], df["humidity"]
df["heat_index"] = (
    -42.379 + 2.04901523 * t + 10.14333127 * rh - 0.22475541 * t * rh
    - 0.00683783 * t ** 2 - 0.05481717 * rh ** 2 + 0.00122874 * t ** 2 * rh
    + 0.00085282 * t * rh ** 2 - 0.00000199 * t ** 2 * rh ** 2
)

# Filter to one city and to temps where the heat index formula is actually valid (>=80F).
df_state = df[(df["city"] == city_filter) & (df["temp_f"] >= 80)].copy()
if df_state.empty:
    raise ValueError(f"No rows found for city {city_filter} with temp_f >= 80 in {len(csv_paths)} CSV files")
print(f"Filtered to city {city_filter}, temp_f >= 80: {df_state.shape[0]} rows")

# Select features and target.
feature_cols = ["temp_f", "humidity", "wind_mph"]
target_col = "heat_index"

X = df_state[feature_cols]
y = df_state[target_col]

# Split train/test for proof of inference.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Standardize features for models that are sensitive to feature scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(random_state=42),
    "Lasso Regression": Lasso(random_state=42),
    "Elastic Net": ElasticNet(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    "Support Vector Regression": SVR(),
    "Neural Network (MLP)": MLPRegressor(random_state=42, max_iter=2000),
}
if xgboost_available:
    models["XGBoost"] = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=100, random_state=42)
if lightgbm_available:
    models["LightGBM"] = lgb.LGBMRegressor(random_state=42, verbose=-1)
if catboost_available:
    models["CatBoost"] = cb.CatBoostRegressor(random_state=42, verbose=False)

# Models that need standardized features (linear/distance/gradient-based); tree
# ensembles split on raw thresholds and don't need scaling.
scaled_models = {"Linear Regression", "Ridge Regression", "Lasso Regression", "Elastic Net",
                  "Support Vector Regression", "Neural Network (MLP)"}

# Use k-fold cross-validation over the full sample to pick a model, since a single
# train/test split on this small a sample is too noisy to trust for that decision.
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
for name, model in models.items():
    X_cv = scaler.fit_transform(X) if name in scaled_models else X
    rmse_scores = -cross_val_score(model, X_cv, y, cv=kfold, scoring="neg_root_mean_squared_error")
    r2_scores = cross_val_score(model, X_cv, y, cv=kfold, scoring="r2")
    cv_results.append({
        "model": name,
        "cv_rmse_mean": rmse_scores.mean(),
        "cv_rmse_std": rmse_scores.std(),
        "cv_r2_mean": r2_scores.mean(),
    })
    print(f"{name}: CV RMSE={rmse_scores.mean():.3f} (+/-{rmse_scores.std():.3f}), CV R2={r2_scores.mean():.3f}")

cv_results_df = pd.DataFrame(cv_results).sort_values("cv_rmse_mean")
print("\nCross-validated model comparison")
print(cv_results_df)

best_model = cv_results_df.iloc[0]
print(f"\nBest model by CV RMSE: {best_model['model']}")

# Fit the chosen model on the train split and predict on the held-out test split,
# as a single proof-of-inference example (not used for model selection).
best_name = best_model["model"]
best_model_obj = models[best_name]
if best_name in scaled_models:
    best_model_obj.fit(X_train_scaled, y_train)
    sample_preds = best_model_obj.predict(X_test_scaled)
else:
    best_model_obj.fit(X_train, y_train)
    sample_preds = best_model_obj.predict(X_test)

sample_output = X_test.copy()
sample_output["actual_heat_index"] = y_test
sample_output["predicted_heat_index"] = sample_preds
print("\nSample test predictions:")
print(sample_output.reset_index(drop=True).head())

heat_bins = [0, 90, 100, 110, 130]
heat_labels = ["cool", "warm", "hot", "very hot"]
sample_output["heat_label"] = pd.cut(sample_output["predicted_heat_index"], bins=heat_bins, labels=heat_labels, right=False)
print("\nExample predicted heat labels:")
print(sample_output[["predicted_heat_index", "heat_label"]].head())


Loaded 137106 rows from 4 CSV files
- data/daily-weather-rice_0_0_0.csv
- data/daily-weather-rice_0_1_0.csv
- data/daily-weather-rice_0_2_0.csv
- data/daily-weather-rice_0_3_0.csv
Filtered to city KQRH, temp_f >= 80: 281 rows
Linear Regression: CV RMSE=1.806 (+/-0.185), CV R2=0.960
Ridge Regression: CV RMSE=1.812 (+/-0.171), CV R2=0.960
Lasso Regression: CV RMSE=3.546 (+/-0.465), CV R2=0.846
Elastic Net: CV RMSE=5.756 (+/-0.453), CV R2=0.597
Random Forest: CV RMSE=1.642 (+/-0.321), CV R2=0.966
Gradient Boosting: CV RMSE=1.388 (+/-0.272), CV R2=0.976
Hist Gradient Boosting: CV RMSE=1.670 (+/-0.415), CV R2=0.964
Support Vector Regression: CV RMSE=2.474 (+/-0.386), CV R2=0.925


/Users/rusli/Desktop/Rice-To-Meet-You/data/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rusli/Desktop/Rice-To-Meet-You/data/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rusli/Desktop/Rice-To-Meet-You/data/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rusli/Desktop/Rice-To-Meet-You/data/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20

Neural Network (MLP): CV RMSE=1.634 (+/-0.454), CV R2=0.966
